# SAR ship benchmark

One notebook for SARATR-X and SAR-JEPA. OpenSARShip 1, NASTaR, and FUSAR-Ship are read in place; no images are copied or reorganized. OpenSARShip uses paired VV/VH images from the dual-polarized GRD archives and exact `Ship.xml` labels.

In [1]:
# Run settings
MODEL = 'saratrx'            # 'saratrx' or 'sarjepa'
RUN_SIZE = 'full'           # 'smoke' or 'full'
TRAINING_MODE = 'full_finetune'  # 'linear_probe', 'partial_finetune', or 'full_finetune'
GOAL = 'nastar'              # 'opensar4', 'opensar6', 'nastar', or 'fusar'

SEEDS = [42] if RUN_SIZE == 'smoke' else [42, 43, 44, 45, 46]
STRICT_REPRODUCIBILITY = True
SAVE_MODELS = RUN_SIZE == 'full'  # full runs always keep the best checkpoint
NUM_WORKERS = 0
SMOKE_SAMPLES_PER_CLASS = 30


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

from datetime import datetime
from functools import partial
from io import BytesIO
from pathlib import Path
import xml.etree.ElementTree as ET
import hashlib
import json
import math
import platform
import random
import re
import shutil
import subprocess
import sys
import time
import traceback
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support, precision_score, recall_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm

start = Path.cwd().resolve()
WORKSPACE_ROOT = next(
    (p for p in (start, *start.parents) if (p / 'SARATR-X').is_dir() and (p / 'SAR-JEPA').is_dir()),
    None,
)
if WORKSPACE_ROOT is None:
    raise FileNotFoundError('Open this notebook from the COMPASS_NET_PAPER workspace.')

choices = {
    'MODEL': (MODEL, {'saratrx', 'sarjepa'}),
    'RUN_SIZE': (RUN_SIZE, {'smoke', 'full'}),
    'TRAINING_MODE': (TRAINING_MODE, {'linear_probe', 'partial_finetune', 'full_finetune'}),
    'GOAL': (GOAL, {'opensar4', 'opensar6', 'nastar', 'fusar'}),
}
for name, (value, allowed) in choices.items():
    if value not in allowed:
        raise ValueError(f'{name} must be one of {sorted(allowed)}')
if not torch.cuda.is_available():
    raise RuntimeError('Select the Python (SARATR-X GPU) kernel and restart the notebook.')

DEVICE = torch.device('cuda:0')
OUTPUT_ROOT = WORKSPACE_ROOT / 'benchmark_outputs'
SESSION_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
USE_AMP = not STRICT_REPRODUCIBILITY

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(STRICT_REPRODUCIBILITY)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

print(f'Workspace: {WORKSPACE_ROOT}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')
print(f'Model: {MODEL} | goal: {GOAL} | size: {RUN_SIZE} | training: {TRAINING_MODE}')


: 

In [ ]:
GOALS = {
    'opensar4': {
        'classes': ('Tanker', 'Fishing', 'Container Ship', 'Bulk Carrier'),
        'image_size': 128, 'transform': 'resize',
    },
    'opensar6': {
        'classes': ('Bulk Carrier', 'Container Ship', 'Dredger', 'Fishing', 'Tanker', 'Tug'),
        'image_size': 128, 'transform': 'resize',
    },
    'nastar': {
        'classes': ('Tanker', 'Fishing', 'Cargo'),
        'image_size': 256, 'transform': 'center_crop',
    },
    'fusar': {
        'classes': ('Tanker', 'General Cargo', 'Fishing', 'Container Ship', 'Bulk Carrier'),
        'image_size': 256, 'transform': 'center_crop',
    },
}

FUSAR_FOLDERS = {
    'Tanker': 'Tanker/Tanker',
    'General Cargo': 'Cargo/GeneralCargo',
    'Fishing': 'Fishing/Fishing',
    'Container Ship': 'Cargo/ContainerShip',
    'Bulk Carrier': 'Cargo/BulkCarrier',
}

def clean_header(value):
    text = str(value).replace('\ufeff', '').lstrip('# ').strip().lower()
    return re.sub(r'[^a-z0-9]+', ' ', text).strip()

def clean_value(value):
    return re.sub(r'\s+', ' ', str(value).strip())

def build_opensar_manifest(goal):
    root = WORKSPACE_ROOT / 'OpenSARShip_1'
    classes = GOALS[goal]['classes']
    label_lookup = {name.casefold(): (index, name) for index, name in enumerate(classes)}
    archives = sorted(root.glob('*GRDH_1SDV*.zip'))
    if not archives:
        raise FileNotFoundError(f'No dual-polarized GRD archives found under {root}')

    records = []
    missing_pairs = []
    member_pattern = re.compile(r'_x(-?\d+)_y(-?\d+)_(vv|vh)\.tiff?$', re.IGNORECASE)
    for archive_path in archives:
        with zipfile.ZipFile(archive_path) as archive:
            members = archive.namelist()
            ship_xml = next((name for name in members if name.lower().endswith('/ship.xml')), None)
            if ship_xml is None:
                raise RuntimeError(f'Ship.xml is missing from {archive_path.name}')
            patch_members = {}
            for member in members:
                if '/Patch_Uint8/' not in member:
                    continue
                match = member_pattern.search(member)
                if match:
                    key = (int(match.group(1)), int(match.group(2)))
                    patch_members.setdefault(key, {})[match.group(3).lower()] = member

            ship_list = ET.fromstring(archive.read(ship_xml))
            for ship in ship_list.findall('.//ship'):
                elaborated_type = clean_value(
                    ship.findtext('./MarineTrafficInformation/Elaborated_type', default='')
                )
                if elaborated_type.casefold() not in label_lookup:
                    continue
                x = int(float(ship.findtext('./SARShipInformation/Center_x')))
                y = int(float(ship.findtext('./SARShipInformation/Center_y')))
                pair = patch_members.get((x, y), {})
                if not {'vv', 'vh'}.issubset(pair):
                    missing_pairs.append(f'{archive_path.name}:{x}_{y}')
                    continue
                label, label_name = label_lookup[elaborated_type.casefold()]
                records.append({
                    'sample_id': f'{archive_path.stem}/{x}_{y}',
                    'source_kind': 'opensar_zip',
                    'archive_path': str(archive_path.resolve()),
                    'vv_member': pair['vv'], 'vh_member': pair['vh'],
                    'label': label, 'label_name': label_name,
                })
    if missing_pairs:
        raise RuntimeError(f'Missing VV/VH data for {len(missing_pairs)} selected OpenSARShip samples.')
    frame = pd.DataFrame(records).drop_duplicates('sample_id')
    return frame.sort_values('sample_id').reset_index(drop=True)

def find_nastar_patch(scene_dir, patch_name):
    patch_dir = scene_dir / 'ship_patches_uint8'
    stem = Path(clean_value(patch_name)).stem
    names = [
        f'{stem}_uint8.tif', f'{stem}.tif', f'{stem}_uint8.tiff', f'{stem}.tiff',
    ]
    if stem.lower().endswith('_uint8'):
        names = [f'{stem}.tif', f'{stem}.tiff', *names]
    return next((patch_dir / name for name in names if (patch_dir / name).is_file()), None)

def build_nastar_manifest():
    root = WORKSPACE_ROOT / 'NovaSAR_dataset'
    classes = GOALS['nastar']['classes']
    label_lookup = {name.casefold(): (i, name) for i, name in enumerate(classes)}
    records = []
    csv_files = sorted(root.rglob('AIS.csv'))
    if not csv_files:
        raise FileNotFoundError(f'No AIS.csv files found under {root}')
    for csv_path in csv_files:
        table = pd.read_csv(csv_path)
        table.columns = [clean_header(name) for name in table.columns]
        required = {'ship type', 'shoreline', 'patch name'}
        if not required.issubset(table.columns):
            continue
        for _, row in table.iterrows():
            ship_type = clean_value(row['ship type'])
            if clean_value(row['shoreline']).casefold() != 'offshore':
                continue
            if ship_type.casefold() not in label_lookup:
                continue
            image_path = find_nastar_patch(csv_path.parent, row['patch name'])
            if image_path is None:
                continue
            label, label_name = label_lookup[ship_type.casefold()]
            records.append({
                'sample_id': image_path.relative_to(root).as_posix(),
                'source_kind': 'file',
                'image_path': str(image_path.resolve()),
                'label': label,
                'label_name': label_name,
            })
    frame = pd.DataFrame(records).drop_duplicates('sample_id')
    return frame.sort_values('sample_id').reset_index(drop=True)

def build_fusar_manifest():
    root = WORKSPACE_ROOT / 'FUSAR_Ship1.0'
    records = []
    for label, label_name in enumerate(GOALS['fusar']['classes']):
        folder = root / FUSAR_FOLDERS[label_name]
        files = sorted((*folder.glob('*.tif'), *folder.glob('*.tiff')))
        if not files:
            raise FileNotFoundError(f'No TIFF images found in {folder}')
        for image_path in files:
            records.append({
                'sample_id': image_path.relative_to(root).as_posix(),
                'source_kind': 'file',
                'image_path': str(image_path.resolve()),
                'label': label,
                'label_name': label_name,
            })
    return pd.DataFrame(records).sort_values('sample_id').reset_index(drop=True)

def build_manifest(goal):
    if goal in {'opensar4', 'opensar6'}:
        return build_opensar_manifest(goal)
    if goal == 'nastar':
        return build_nastar_manifest()
    if goal == 'fusar':
        return build_fusar_manifest()
    raise ValueError(f'Unknown goal: {goal}')

class ShipDataset(Dataset):
    def __init__(self, frame, image_size, transform_name):
        self.frame = frame.reset_index(drop=True)
        self.image_size = image_size
        self.transform_name = transform_name
        self.archives = {}

    def __len__(self):
        return len(self.frame)

    def transform(self, image):
        if self.transform_name == 'center_crop':
            image = TF.center_crop(image, [self.image_size, self.image_size])
        else:
            image = TF.resize(
                image, [self.image_size, self.image_size],
                interpolation=InterpolationMode.BILINEAR, antialias=True,
            )
        return TF.to_tensor(image)

    def get_archive(self, path):
        if path not in self.archives:
            self.archives[path] = zipfile.ZipFile(path)
        return self.archives[path]

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        if row.source_kind == 'opensar_zip':
            archive = self.get_archive(row.archive_path)
            channels = []
            for member in (row.vv_member, row.vh_member):
                with Image.open(BytesIO(archive.read(member))) as source:
                    channels.append(self.transform(source.convert('L')))
            image = torch.cat(channels, dim=0)
        else:
            with Image.open(row.image_path) as source:
                image = self.transform(source.convert('L'))
        return image, int(row.label), row.sample_id

    def __del__(self):
        for archive in self.archives.values():
            archive.close()

def split_manifest(manifest, seed, run_size):
    selected = []
    for label, group in manifest.groupby('label', sort=True):
        group = group.sort_values('sample_id').reset_index(drop=True)
        rng = np.random.default_rng(seed + int(label))
        order = rng.permutation(len(group))
        count = min(SMOKE_SAMPLES_PER_CLASS, len(group)) if run_size == 'smoke' else len(group)
        selected.append(group.iloc[order[:count]])
    selected = pd.concat(selected, ignore_index=True)

    parts = []
    for label, group in selected.groupby('label', sort=True):
        group = group.sort_values('sample_id').reset_index(drop=True)
        rng = np.random.default_rng(seed + 10_000 + int(label))
        group = group.iloc[rng.permutation(len(group))].copy()
        n_val = max(1, int(math.floor(len(group) * 0.10 + 0.5)))
        n_test = max(1, int(math.floor(len(group) * 0.20 + 0.5)))
        n_train = len(group) - n_val - n_test
        if n_train < 1:
            raise ValueError(f'Class {label} has too few samples for a 70/10/20 split.')
        group['split'] = ['train'] * n_train + ['validation'] * n_val + ['test'] * n_test
        parts.append(group)
    return pd.concat(parts, ignore_index=True).sort_values('sample_id').reset_index(drop=True)

def save_split_manifest(frame, seed, folder):
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f'seed_{seed}.csv'
    columns = ['sample_id', 'label_name', 'label', 'split']
    current = frame[columns].sort_values('sample_id').reset_index(drop=True)
    if path.is_file():
        saved = pd.read_csv(path)[columns].sort_values('sample_id').reset_index(drop=True)
        if not saved.equals(current):
            raise RuntimeError(f'Saved split does not match the current dataset: {path}')
    else:
        current.to_csv(path, index=False)
    return path

def make_loaders(frame, batch_size, seed, spec):
    generator = torch.Generator().manual_seed(seed)
    common = {
        'batch_size': batch_size, 'num_workers': NUM_WORKERS,
        'pin_memory': True, 'persistent_workers': NUM_WORKERS > 0,
    }
    loaders = {}
    for split in ('train', 'validation', 'test'):
        dataset = ShipDataset(
            frame[frame.split == split], spec['image_size'], spec['transform'],
        )
        loaders[split] = DataLoader(
            dataset, shuffle=split == 'train',
            generator=generator if split == 'train' else None, **common,
        )
    return loaders


: 

In [ ]:
spec = GOALS[GOAL]
CLASS_NAMES = list(spec['classes'])
IMAGE_SIZE = spec['image_size']
INPUT_CHANNELS = 2 if GOAL in {'opensar4', 'opensar6'} else 1
MODEL_INPUT_CHANNELS = INPUT_CHANNELS if INPUT_CHANNELS == 2 else (3 if MODEL == 'saratrx' else 1)
manifest = build_manifest(GOAL)
if set(manifest.label_name) != set(CLASS_NAMES):
    raise RuntimeError('The indexed classes do not match the selected goal.')

fingerprint_text = '\n'.join(
    f'{row.sample_id}|{row.label_name}' for row in manifest.itertuples(index=False)
)
DATASET_FINGERPRINT = hashlib.sha256(fingerprint_text.encode()).hexdigest()

common = {
    ('saratrx', 'linear_probe'): dict(batch_size=32, epochs=50, head_lr=1e-3, backbone_lr=0.0, weight_decay=1e-4, betas=(0.9, 0.95), warmup_epochs=2),
    ('saratrx', 'partial_finetune'): dict(batch_size=16, epochs=50, head_lr=3e-4, backbone_lr=1e-5, weight_decay=1e-4, betas=(0.9, 0.95), warmup_epochs=3),
    ('saratrx', 'full_finetune'): dict(batch_size=8, epochs=50, head_lr=1e-4, backbone_lr=3e-6, weight_decay=1e-4, betas=(0.9, 0.95), warmup_epochs=5),
    ('sarjepa', 'linear_probe'): dict(batch_size=32, epochs=50, head_lr=1e-3, backbone_lr=0.0, weight_decay=5e-4, betas=(0.9, 0.999), warmup_epochs=2),
    ('sarjepa', 'partial_finetune'): dict(batch_size=16, epochs=50, head_lr=1e-3, backbone_lr=1e-5, weight_decay=5e-4, betas=(0.9, 0.999), warmup_epochs=3),
    ('sarjepa', 'full_finetune'): dict(batch_size=8, epochs=50, head_lr=1e-4, backbone_lr=1e-5, weight_decay=5e-4, betas=(0.9, 0.999), warmup_epochs=5),
}
HP = common[(MODEL, TRAINING_MODE)].copy()
if RUN_SIZE == 'smoke':
    HP.update(batch_size=7, epochs=2, warmup_epochs=1)

display(manifest.label_name.value_counts().reindex(CLASS_NAMES).rename('samples').to_frame())
print(f'Total samples: {len(manifest)}')
print(f'Image size: {IMAGE_SIZE} x {IMAGE_SIZE} ({spec["transform"]})')
print(f'Input channels: {INPUT_CHANNELS}' + (' (VV, VH)' if INPUT_CHANNELS == 2 else ''))
print(f'Dataset fingerprint: {DATASET_FINGERPRINT}')
print(f'Hyperparameters: {HP}')

preview = ShipDataset(manifest, IMAGE_SIZE, spec['transform'])
fig, axes = plt.subplots(INPUT_CHANNELS, len(CLASS_NAMES), figsize=(3.2 * len(CLASS_NAMES), 3.2 * INPUT_CHANNELS), squeeze=False)
for label, name in enumerate(CLASS_NAMES):
    index = manifest.index[manifest.label == label][0]
    image, _, _ = preview[index]
    for channel in range(INPUT_CHANNELS):
        ax = axes[channel, label]
        ax.imshow(image[channel], cmap='gray', vmin=0, vmax=1)
        ax.set_title(name if channel == 0 else '')
        if label == 0:
            ax.set_ylabel(('VV', 'VH')[channel] if INPUT_CHANNELS == 2 else 'Intensity')
        ax.set_xticks([])
        ax.set_yticks([])
plt.tight_layout()


: 

In [ ]:
def sha256_file(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        while chunk := handle.read(block_size):
            digest.update(chunk)
    return digest.hexdigest()

def adapt_input_projection(source, target):
    key = 'patch_embed.proj.weight'
    if key not in source or key not in target or source[key].shape == target[key].shape:
        return
    old = source[key]
    new = target[key]
    if old.ndim != 4 or old.shape[0] != new.shape[0] or old.shape[2:] != new.shape[2:]:
        return
    source[key] = old.sum(dim=1, keepdim=True).repeat(1, new.shape[1], 1, 1) / new.shape[1]

def load_saratrx_checkpoint(model, path):
    checkpoint = torch.load(path, map_location='cpu')
    source = checkpoint.get('model', checkpoint)
    source = {key.removeprefix('module.'): value for key, value in source.items()}
    target = model.state_dict()
    adapt_input_projection(source, target)
    key = 'absolute_pos_embed'
    if key in source and key in target and source[key].shape != target[key].shape:
        old = source[key]
        old_side = int(old.shape[1] ** 0.5)
        new_side = int(target[key].shape[1] ** 0.5)
        old = old.reshape(1, old_side, old_side, old.shape[-1]).permute(0, 3, 1, 2)
        old = F.interpolate(old, size=(new_side, new_side), mode='bicubic', align_corners=False)
        source[key] = old.permute(0, 2, 3, 1).flatten(1, 2)
    compatible = {key: value for key, value in source.items() if key in target and value.shape == target[key].shape}
    message = model.load_state_dict(compatible, strict=False)
    return {'loaded': len(compatible), 'missing': tuple(message.missing_keys)}

def build_saratrx():
    model_path = WORKSPACE_ROOT / 'SARATR-X' / 'classification'
    sys.path.insert(0, str(model_path))
    from model.hivit import HiViT
    model = HiViT(
        img_size=IMAGE_SIZE, patch_size=16, embed_dim=512, depths=[2, 2, 20],
        num_heads=8, stem_mlp_ratio=3.0, in_chans=MODEL_INPUT_CHANNELS, mlp_ratio=4.0,
        num_classes=len(CLASS_NAMES), ape=True, rpe=False,
        norm_layer=partial(nn.LayerNorm, eps=1e-6),
    )
    path = WORKSPACE_ROOT / 'SARATR-X' / 'checkpoints' / 'checkpoint-200.pth'
    if not path.is_file():
        raise FileNotFoundError(path)
    return model, path, load_saratrx_checkpoint(model, path)

def load_sarjepa_checkpoint(model, path):
    checkpoint = torch.load(path, map_location='cpu')
    source = checkpoint.get('model', checkpoint)
    source = {key.removeprefix('module.'): value for key, value in source.items()}
    target = model.state_dict()
    adapt_input_projection(source, target)
    compatible = {key: value for key, value in source.items() if key in target and value.shape == target[key].shape}
    message = model.load_state_dict(compatible, strict=False)
    missing = set(message.missing_keys) - {'head.weight', 'head.bias'}
    if missing:
        raise RuntimeError(f'Missing SAR-JEPA encoder tensors: {sorted(missing)}')
    return {'loaded': len(compatible), 'missing': tuple(message.missing_keys)}

def build_sarjepa():
    model_path = WORKSPACE_ROOT / 'SAR-JEPA' / 'few_shot_classification' / 'finetune'
    sys.path.insert(0, str(model_path))
    from vision_transformer_irpe import VisionTransformer
    model = VisionTransformer(
        img_size=IMAGE_SIZE, patch_size=16, embed_dim=768, in_chans=MODEL_INPUT_CHANNELS,
        num_classes=len(CLASS_NAMES), depth=12, num_heads=12, mlp_ratio=4,
        qkv_bias=True, norm_layer=partial(nn.LayerNorm, eps=1e-6),
    )
    path = WORKSPACE_ROOT / 'SAR-JEPA' / 'checkpoints' / 'weights' / 'SAR-JEPA' / 'checkpoint-200.pth'
    if not path.is_file():
        raise FileNotFoundError(path)
    return model, path, load_sarjepa_checkpoint(model, path)

def build_model():
    return build_saratrx() if MODEL == 'saratrx' else build_sarjepa()

def configure_model(model):
    if TRAINING_MODE in {'linear_probe', 'partial_finetune'}:
        head = model.head
        nn.init.trunc_normal_(head.weight, std=0.01)
        nn.init.zeros_(head.bias)
        model.head = nn.Sequential(nn.BatchNorm1d(head.in_features, affine=False, eps=1e-6), head)

    if TRAINING_MODE == 'full_finetune':
        for parameter in model.parameters():
            parameter.requires_grad = True
        return model

    for parameter in model.parameters():
        parameter.requires_grad = False
    for parameter in model.head.parameters():
        parameter.requires_grad = True

    norm = model.fc_norm if MODEL == 'saratrx' else model.norm
    if MODEL == 'saratrx':
        for parameter in norm.parameters():
            parameter.requires_grad = True
    if TRAINING_MODE == 'partial_finetune':
        for block in model.blocks[-4:]:
            for parameter in block.parameters():
                parameter.requires_grad = True
        for parameter in norm.parameters():
            parameter.requires_grad = True
    return model

def clear_rpe_cache(model):
    for module in model.modules():
        if hasattr(module, '_rp_bucket_buf'):
            module._rp_bucket_buf = (None, None, None)

def prepare_images(images):
    images = images.to(DEVICE, non_blocking=True)
    return images.repeat(1, 3, 1, 1) if MODEL_INPUT_CHANNELS == 3 else images

def optimizer_groups(model):
    norm = model.fc_norm if MODEL == 'saratrx' else model.norm
    head = [p for p in model.head.parameters() if p.requires_grad]
    final_norm = [p for p in norm.parameters() if p.requires_grad]
    head_ids = {id(p) for p in (*head, *final_norm)}
    backbone = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    groups = [{'params': [*head, *final_norm], 'lr': HP['head_lr'], 'name': 'head'}]
    if backbone:
        groups.append({'params': backbone, 'lr': HP['backbone_lr'], 'name': 'backbone'})
    return groups


: 

In [ ]:
METRICS = ('OA', 'AA', 'APr', 'AF1')

def calculate_metrics(truth, prediction):
    labels = list(range(len(CLASS_NAMES)))
    return {
        'OA': accuracy_score(truth, prediction),
        'AA': recall_score(truth, prediction, labels=labels, average='macro', zero_division=0),
        'APr': precision_score(truth, prediction, labels=labels, average='macro', zero_division=0),
        'AF1': f1_score(truth, prediction, labels=labels, average='macro', zero_division=0),
    }

def calculate_per_class_metrics(truth, prediction):
    labels = list(range(len(CLASS_NAMES)))
    precision, recall, f1, support = precision_recall_fscore_support(
        truth, prediction, labels=labels, zero_division=0,
    )
    return pd.DataFrame({
        'class': CLASS_NAMES, 'precision': precision, 'recall': recall,
        'f1': f1, 'support': support.astype(int),
    })

def evaluate(model, loader, criterion):
    model.eval()
    truth, prediction, probabilities, sample_ids = [], [], [], []
    loss_total = 0.0
    with torch.inference_mode():
        for images, labels, ids in loader:
            inputs = prepare_images(images)
            targets = labels.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
                logits = model(inputs)
                loss = criterion(logits, targets)
            loss_total += loss.item() * labels.size(0)
            truth.extend(labels.tolist())
            prediction.extend(logits.argmax(1).cpu().tolist())
            probabilities.append(logits.softmax(dim=1).cpu().numpy())
            sample_ids.extend(ids)
    truth = np.asarray(truth)
    prediction = np.asarray(prediction)
    probabilities = np.concatenate(probabilities, axis=0)
    return calculate_metrics(truth, prediction), loss_total / len(truth), truth, prediction, probabilities, sample_ids

def schedule_factor(epoch):
    warmup = HP['warmup_epochs']
    if epoch < warmup:
        return (epoch + 1) / max(warmup, 1)
    progress = (epoch - warmup) / max(HP['epochs'] - warmup - 1, 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

def format_duration(seconds):
    seconds = int(round(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d}'

def make_prediction_table(sample_ids, truth, prediction, probabilities):
    table = pd.DataFrame({
        'sample_id': sample_ids,
        'true_label': [CLASS_NAMES[value] for value in truth],
        'predicted_label': [CLASS_NAMES[value] for value in prediction],
        'correct': truth == prediction,
        'confidence': probabilities.max(axis=1),
    })
    for index, name in enumerate(CLASS_NAMES):
        column = 'probability_' + re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')
        table[column] = probabilities[:, index]
    return table

def save_confusion_outputs(truth, prediction, run_dir):
    labels = list(range(len(CLASS_NAMES)))
    matrices = {
        'Raw counts': confusion_matrix(truth, prediction, labels=labels),
        'Row-normalized recall': confusion_matrix(truth, prediction, labels=labels, normalize='true'),
        'Column-normalized precision': confusion_matrix(truth, prediction, labels=labels, normalize='pred'),
    }
    lines = ['Rows are true classes; columns are predicted classes.', '']
    for title, matrix in matrices.items():
        frame = pd.DataFrame(matrix, index=CLASS_NAMES, columns=CLASS_NAMES)
        lines.extend([title, frame.to_string(float_format=lambda value: f'{value:.4f}'), ''])
    (run_dir / 'CONFUSION_MATRICES.txt').write_text('\n'.join(lines), encoding='utf-8')

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (title, matrix) in zip(axes, matrices.items()):
        ax.imshow(matrix, cmap='Blues')
        threshold = matrix.max() / 2 if matrix.size else 0
        for row in range(matrix.shape[0]):
            for column in range(matrix.shape[1]):
                value = matrix[row, column]
                label = f'{int(value)}' if title == 'Raw counts' else f'{value:.2f}'
                ax.text(column, row, label, ha='center', va='center', color='white' if value > threshold else 'black')
        ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha='right')
        ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
        ax.set_title(title)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
    plt.tight_layout()
    fig.savefig(run_dir / 'confusion_matrices.png', dpi=180, bbox_inches='tight')
    plt.close(fig)
    return matrices['Raw counts'].tolist()

def train_one_seed(seed, loaders, run_dir):
    set_seed(seed)
    torch.cuda.reset_peak_memory_stats()
    model, checkpoint_path, checkpoint_report = build_model()
    model = configure_model(model).to(DEVICE)
    clear_rpe_cache(model)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        optimizer_groups(model), betas=HP['betas'], weight_decay=HP['weight_decay'],
    )
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, schedule_factor)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    writer = SummaryWriter(log_dir=str(run_dir / 'tensorboard'), flush_secs=2)

    best_state = None
    best_epoch = 0
    best_af1 = -1.0
    best_validation_metrics = None
    best_validation_loss = None
    history = []
    global_step = 0
    started = time.perf_counter()
    progress = tqdm(total=HP['epochs'] * len(loaders['train']), desc=f'{MODEL} seed {seed}', unit='batch')

    for epoch in range(HP['epochs']):
        epoch_started = time.perf_counter()
        model.train()
        loss_total = 0.0
        correct = 0
        seen = 0
        for images, labels, _ in loaders['train']:
            inputs = prepare_images(images)
            targets = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
                logits = model(inputs)
                loss = criterion(logits, targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_total += loss.item() * labels.size(0)
            correct += (logits.argmax(1) == targets).sum().item()
            seen += labels.size(0)
            writer.add_scalar('loss/train_batch', loss.item(), global_step)
            global_step += 1
            progress.update(1)
            progress.set_postfix(epoch=f'{epoch + 1}/{HP["epochs"]}', loss=f'{loss.item():.4f}')

        val_metrics, val_loss, _, _, _, _ = evaluate(model, loaders['validation'], criterion)
        elapsed = time.perf_counter() - started
        remaining = elapsed / (epoch + 1) * (HP['epochs'] - epoch - 1)
        row = {
            'epoch': epoch + 1, 'train_loss': loss_total / seen, 'train_OA': correct / seen,
            'validation_loss': val_loss, 'epoch_seconds': time.perf_counter() - epoch_started,
            'estimated_remaining_seconds': remaining,
            **{f'validation_{name}': value for name, value in val_metrics.items()},
        }
        history.append(row)
        writer.add_scalar('loss/train_epoch', row['train_loss'], epoch + 1)
        writer.add_scalar('loss/validation', val_loss, epoch + 1)
        writer.add_scalar('metrics/train_OA', row['train_OA'], epoch + 1)
        for name, value in val_metrics.items():
            writer.add_scalar(f'metrics/validation_{name}', value, epoch + 1)
        for group in optimizer.param_groups:
            writer.add_scalar(f'learning_rate/{group["name"]}', group['lr'], epoch + 1)
        writer.add_scalar('timing/epoch_seconds', row['epoch_seconds'], epoch + 1)
        writer.add_scalar('timing/estimated_remaining_seconds', remaining, epoch + 1)
        writer.add_scalar('gpu/peak_memory_MB', torch.cuda.max_memory_allocated() / 1024**2, epoch + 1)
        writer.flush()
        if val_metrics['AF1'] > best_af1:
            best_af1 = val_metrics['AF1']
            best_epoch = epoch + 1
            best_validation_metrics = val_metrics.copy()
            best_validation_loss = val_loss
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        scheduler.step()

    progress.close()
    model.load_state_dict(best_state)
    test_metrics, test_loss, truth, prediction, probabilities, sample_ids = evaluate(model, loaders['test'], criterion)
    for name, value in test_metrics.items():
        writer.add_scalar(f'test/{name}', value, best_epoch)
    writer.add_scalar('test/cross_entropy', test_loss, best_epoch)
    writer.close()

    history = pd.DataFrame(history)
    predictions = make_prediction_table(sample_ids, truth, prediction, probabilities)
    per_class = calculate_per_class_metrics(truth, prediction)
    report = {
        'seed': seed, 'best_epoch': best_epoch, 'best_validation_loss': best_validation_loss,
        'test_loss': test_loss, 'elapsed_seconds': time.perf_counter() - started,
        'peak_cuda_memory_MB': torch.cuda.max_memory_allocated() / 1024**2,
        'checkpoint': str(checkpoint_path), 'checkpoint_sha256': SOURCE_CHECKPOINT_SHA256,
        'checkpoint_tensors_loaded': checkpoint_report['loaded'],
        **{f'best_validation_{name}': value for name, value in best_validation_metrics.items()},
        **test_metrics,
    }
    if SAVE_MODELS:
        model_path = run_dir / 'best_model.pth'
        torch.save({
            'model_state_dict': best_state, 'model': MODEL, 'goal': GOAL,
            'training_mode': TRAINING_MODE, 'classes': CLASS_NAMES,
            'image_size': IMAGE_SIZE, 'input_channels': INPUT_CHANNELS,
            'polarizations': ['VV', 'VH'] if INPUT_CHANNELS == 2 else None,
            'seed': seed, 'best_epoch': best_epoch,
            'hyperparameters': HP, 'dataset_fingerprint': DATASET_FINGERPRINT,
        }, model_path)
        report['trained_model'] = str(model_path)
        report['trained_model_sha256'] = sha256_file(model_path)
    return report, history, predictions, per_class, truth, prediction


: 

In [ ]:
folder_name = f'{SESSION_ID}_{GOAL}_{MODEL}_{TRAINING_MODE}'
session_root = OUTPUT_ROOT / folder_name if RUN_SIZE == 'full' else OUTPUT_ROOT / 'smoke' / folder_name
runs_root = session_root / 'runs'
splits_root = session_root / 'splits'
runs_root.mkdir(parents=True, exist_ok=True)
splits_root.mkdir(parents=True, exist_ok=True)
(session_root / 'STATUS.txt').write_text('RUNNING\n', encoding='utf-8')
experiment_started = time.perf_counter()

SOURCE_CHECKPOINT = (
    WORKSPACE_ROOT / 'SARATR-X' / 'checkpoints' / 'checkpoint-200.pth'
    if MODEL == 'saratrx'
    else WORKSPACE_ROOT / 'SAR-JEPA' / 'checkpoints' / 'weights' / 'SAR-JEPA' / 'checkpoint-200.pth'
)
SOURCE_CHECKPOINT_SHA256 = sha256_file(SOURCE_CHECKPOINT)
SOURCE_REPOSITORY = WORKSPACE_ROOT / ('SARATR-X' if MODEL == 'saratrx' else 'SAR-JEPA')
source_commit = subprocess.run(
    ['git', '-C', str(SOURCE_REPOSITORY), 'rev-parse', 'HEAD'],
    capture_output=True, text=True, check=False,
).stdout.strip()
source_status = subprocess.run(
    ['git', '-C', str(SOURCE_REPOSITORY), 'status', '--short'],
    capture_output=True, text=True, check=False,
).stdout.strip()
source_diff = subprocess.run(
    ['git', '-C', str(SOURCE_REPOSITORY), 'diff', '--binary', '--no-ext-diff'],
    capture_output=True, text=True, check=False,
).stdout
(session_root / 'source_changes.patch').write_text(source_diff, encoding='utf-8')
(session_root / 'source_repository.txt').write_text(
    f'Repository: {SOURCE_REPOSITORY.name}\nCommit: {source_commit or "unavailable"}\nStatus:\n{source_status or "clean"}\n',
    encoding='utf-8',
)

if GOAL in {'opensar4', 'opensar6'}:
    dataset_table = manifest[
        ['sample_id', 'label_name', 'label', 'source_kind', 'archive_path', 'vv_member', 'vh_member']
    ].copy()
    file_sizes = pd.Series(index=dataset_table.index, dtype='int64')
    hashes = pd.Series('', index=dataset_table.index, dtype='object')
    for archive_path, group in tqdm(
        dataset_table.groupby('archive_path', sort=True), desc='Reading OpenSARShip archives', unit='archive',
    ):
        with zipfile.ZipFile(archive_path) as archive:
            for index, row in group.iterrows():
                file_sizes.loc[index] = sum(archive.getinfo(member).file_size for member in (row.vv_member, row.vh_member))
                if RUN_SIZE == 'full':
                    digest = hashlib.sha256()
                    for member in (row.vv_member, row.vh_member):
                        digest.update(archive.read(member))
                        digest.update(b'\0')
                    hashes.loc[index] = digest.hexdigest()
    dataset_table['file_size_bytes'] = file_sizes.astype('int64')
    dataset_table['sha256'] = hashes
    dataset_table['archive_path'] = dataset_table.archive_path.map(
        lambda path: Path(path).relative_to(WORKSPACE_ROOT).as_posix()
    )
else:
    dataset_table = manifest[['sample_id', 'label_name', 'label', 'source_kind', 'image_path']].copy()
    dataset_table['file_size_bytes'] = dataset_table.image_path.map(lambda path: Path(path).stat().st_size)
    dataset_table['sha256'] = (
        [sha256_file(Path(path)) for path in tqdm(dataset_table.image_path, desc='Hashing dataset', unit='image')]
        if RUN_SIZE == 'full' else ''
    )
    dataset_table['image_path'] = dataset_table.image_path.map(
        lambda path: Path(path).relative_to(WORKSPACE_ROOT).as_posix()
    )
if RUN_SIZE == 'full':
    fingerprint_text = '\n'.join(
        f'{row.sample_id}|{row.label_name}|{row.sha256}' for row in dataset_table.itertuples(index=False)
    )
    DATASET_FINGERPRINT = hashlib.sha256(fingerprint_text.encode()).hexdigest()
dataset_table.to_csv(session_root / 'dataset_manifest.csv', index=False)

notebook_path = WORKSPACE_ROOT / 'sar_ship_benchmark.ipynb'
if notebook_path.is_file():
    shutil.copy2(notebook_path, session_root / 'notebook_used.ipynb')
freeze = subprocess.run(
    [sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, check=False,
)
(session_root / 'requirements.txt').write_text(freeze.stdout, encoding='utf-8')
try:
    driver = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader'],
        capture_output=True, text=True, check=False,
    ).stdout.strip() or 'unavailable'
except FileNotFoundError:
    driver = 'unavailable'
environment_lines = [
    f'Platform: {platform.platform()}',
    f'Python: {platform.python_version()}',
    f'Python executable: {sys.executable}',
    f'PyTorch: {torch.__version__}',
    f'CUDA runtime: {torch.version.cuda}',
    f'cuDNN: {torch.backends.cudnn.version()}',
    f'GPU and driver: {driver}',
    f'Deterministic algorithms: {STRICT_REPRODUCIBILITY}',
    f'Automatic mixed precision: {USE_AMP}',
    f'Number of workers: {NUM_WORKERS}',
    f'Source repository: {SOURCE_REPOSITORY.name}',
    f'Source commit: {source_commit or "unavailable"}',
]
(session_root / 'environment.txt').write_text('\n'.join(environment_lines) + '\n', encoding='utf-8')
protocol_lines = [
    f'Dataset goal: {GOAL}',
    f'Classes: {", ".join(CLASS_NAMES)}',
    f'Input: {IMAGE_SIZE} x {IMAGE_SIZE}, {spec["transform"]}',
    f'Input channels: {"VV, VH" if INPUT_CHANNELS == 2 else "single-channel intensity"}',
    'Pretrained input projection: channel-summed and evenly distributed across VV/VH' if INPUT_CHANNELS == 2 else 'Pretrained input projection: unchanged',
    'Split: class-stratified 70% training, 10% validation, 20% testing',
    f'Repeated runs: {len(SEEDS)} independent seeded splits ({", ".join(map(str, SEEDS))})',
    'Model selection: highest validation AF1; test set evaluated once with that checkpoint',
    'OA: overall accuracy',
    'AA: macro-average recall',
    'APr: macro-average precision',
    'AF1: macro-average F1-score',
    'Final result: arithmetic mean ± sample standard deviation across runs',
]
(session_root / 'protocol.txt').write_text('\n'.join(protocol_lines) + '\n', encoding='utf-8')
config = {
    'created_at': datetime.now().astimezone().isoformat(), 'model': MODEL, 'goal': GOAL,
    'run_size': RUN_SIZE, 'training_mode': TRAINING_MODE, 'classes': CLASS_NAMES,
    'image_size': IMAGE_SIZE, 'image_transform': spec['transform'],
    'input_channels': INPUT_CHANNELS,
    'polarizations': ['VV', 'VH'] if INPUT_CHANNELS == 2 else None,
    'input_projection_adaptation': (
        'sum pretrained input-channel weights and distribute equally across VV/VH'
        if INPUT_CHANNELS == 2 else 'none'
    ),
    'split': {'training': 0.70, 'validation': 0.10, 'testing': 0.20},
    'seeds': SEEDS, 'hyperparameters': HP,
    'strict_reproducibility': STRICT_REPRODUCIBILITY, 'save_models': SAVE_MODELS,
    'dataset_fingerprint': DATASET_FINGERPRINT,
    'source_checkpoint': str(SOURCE_CHECKPOINT.relative_to(WORKSPACE_ROOT)),
    'source_checkpoint_sha256': SOURCE_CHECKPOINT_SHA256,
    'source_repository': SOURCE_REPOSITORY.name, 'source_commit': source_commit,
    'source_worktree_status': source_status.splitlines(),
}
with open(session_root / 'config.json', 'w', encoding='utf-8') as handle:
    json.dump(config, handle, indent=2)

results = []
for seed in SEEDS:
    run_dir = runs_root / f'seed_{seed}'
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / 'STATUS.txt').write_text('RUNNING\n', encoding='utf-8')
    try:
        split_frame = split_manifest(manifest, seed, RUN_SIZE)
        split_path = save_split_manifest(split_frame, seed, splits_root)
        loaders = make_loaders(split_frame, HP['batch_size'], seed, spec)
        counts = split_frame.groupby(['split', 'label_name']).size().unstack(fill_value=0)
        counts = counts.reindex(index=['train', 'validation', 'test'], columns=CLASS_NAMES, fill_value=0)
        counts.to_csv(run_dir / 'split_counts.csv')
        display(counts)

        report, history, predictions, per_class, truth, prediction = train_one_seed(seed, loaders, run_dir)
        report['split_manifest'] = str(split_path.relative_to(session_root))
        report['dataset_fingerprint'] = DATASET_FINGERPRINT
        report['confusion_matrix'] = save_confusion_outputs(truth, prediction, run_dir)
        results.append(report)

        history.to_csv(run_dir / 'history.csv', index=False)
        predictions.to_csv(run_dir / 'test_predictions.csv', index=False)
        per_class.to_csv(run_dir / 'per_class_metrics.csv', index=False)
        (run_dir / 'training.log').write_text(history.to_string(index=False) + '\n', encoding='utf-8')

        metric_rows = pd.DataFrame({
            'Metric': [*METRICS, 'Cross-entropy'],
            'Validation': [*(100 * report[f'best_validation_{name}'] for name in METRICS), report['best_validation_loss']],
            'Test': [*(100 * report[name] for name in METRICS), report['test_loss']],
            'Unit': ['%' for _ in METRICS] + ['loss'],
        })
        class_text = per_class.copy()
        for column in ('precision', 'recall', 'f1'):
            class_text[column] = class_text[column] * 100
        metrics_lines = [
            f'Seed: {seed}', f'Best epoch: {report["best_epoch"]}',
            f'Best validation AF1: {100 * report["best_validation_AF1"]:.2f}%',
            f'Training time: {format_duration(report["elapsed_seconds"])}',
            f'Peak CUDA memory: {report["peak_cuda_memory_MB"]:.1f} MB', '',
            'Test metrics', metric_rows.to_string(index=False, float_format=lambda value: f'{value:.4f}'), '',
            'Per-class test metrics (%)',
            class_text.to_string(index=False, float_format=lambda value: f'{value:.2f}'), '',
            'Split counts', counts.to_string(), '',
        ]
        (run_dir / 'METRICS.txt').write_text('\n'.join(metrics_lines), encoding='utf-8')
        run_config = {**config, **report, 'split_counts': counts.to_dict()}
        with open(run_dir / 'metrics.json', 'w', encoding='utf-8') as handle:
            json.dump({**report, 'per_class': per_class.to_dict(orient='records')}, handle, indent=2)
        with open(run_dir / 'run_config.json', 'w', encoding='utf-8') as handle:
            json.dump(run_config, handle, indent=2)
        (run_dir / 'STATUS.txt').write_text('COMPLETE\n', encoding='utf-8')
        print(pd.Series({key: report[key] for key in ('seed', 'best_epoch', *METRICS)}))
    except Exception:
        error_text = traceback.format_exc()
        (run_dir / 'ERROR.txt').write_text(error_text, encoding='utf-8')
        (run_dir / 'STATUS.txt').write_text('FAILED\n', encoding='utf-8')
        (session_root / 'STATUS.txt').write_text('FAILED\n', encoding='utf-8')
        raise
    finally:
        torch.cuda.empty_cache()

results = pd.DataFrame(results)
results.to_csv(session_root / 'results.csv', index=False)
summary = results[list(METRICS)].agg(['mean', 'std']).T * 100
summary['std'] = summary['std'].fillna(0.0)
summary['mean ± std (%)'] = summary.apply(lambda row: f'{row["mean"]:.2f} ± {row["std"]:.2f}', axis=1)
summary.to_csv(session_root / 'summary.csv')

run_table = results[['seed', *METRICS, 'best_epoch', 'elapsed_seconds']].copy()
for metric in METRICS:
    run_table[metric] = 100 * run_table[metric]
run_table['time'] = run_table.elapsed_seconds.map(format_duration)
run_table = run_table.drop(columns='elapsed_seconds')
summary_table = summary.rename_axis('Metric').reset_index()[['Metric', 'mean', 'std', 'mean ± std (%)']]
total_seconds = time.perf_counter() - experiment_started
summary_lines = [
    'SAR ship classification experiment', '',
    f'Goal: {GOAL}', f'Model: {MODEL}', f'Training mode: {TRAINING_MODE}',
    f'Runs: {len(results)}', f'Seeds: {", ".join(map(str, SEEDS))}',
    f'Dataset fingerprint: {DATASET_FINGERPRINT}',
    f'Total time: {format_duration(total_seconds)}', '',
    'Results by run (%)',
    run_table.to_string(index=False, float_format=lambda value: f'{value:.2f}'), '',
    'Final metrics: mean ± sample standard deviation (%)',
    summary_table.to_string(index=False, float_format=lambda value: f'{value:.2f}'), '',
]
(session_root / 'SUMMARY.txt').write_text('\n'.join(summary_lines), encoding='utf-8')
(session_root / 'STATUS.txt').write_text('COMPLETE\n', encoding='utf-8')

checksum_lines = [f'{SOURCE_CHECKPOINT_SHA256}  SOURCE_CHECKPOINT::{SOURCE_CHECKPOINT.relative_to(WORKSPACE_ROOT)}']
for path in sorted(session_root.rglob('*')):
    if path.is_file() and path.name != 'checksums.sha256' and 'tensorboard' not in path.parts:
        checksum_lines.append(f'{sha256_file(path)}  {path.relative_to(session_root)}')
(session_root / 'checksums.sha256').write_text('\n'.join(checksum_lines) + '\n', encoding='utf-8')

display(summary[['mean ± std (%)']])
print(f'Results: {session_root}')
print(f'TensorBoard: tensorboard --logdir {runs_root} --port 6006 --reload_interval 2')


: 

## Reproducible runs

Use **Restart Kernel and Run All Cells** after changing the configuration. Full runs use five fixed seeds and report OA, AA, APr, and AF1 as mean ± sample standard deviation. Each full experiment is saved as `benchmark_outputs/<timestamp>_<goal>_<model>_<training_mode>/`. Its `SUMMARY.txt` contains the readable final table. Each `runs/seed_*` folder contains the best trained model, metrics, class-level results, predictions with probabilities, confusion matrices, training history, log, status, and TensorBoard events. The experiment folder also keeps the exact splits, dataset hashes, configuration, environment, package versions, notebook snapshot, protocol, and artifact checksums.

For the closest match on another computer, copy the complete experiment folder, the same dataset release, and the pretrained checkpoint. Recreate `requirements.txt`, keep the same GPU model and driver, and use the saved split files and settings. Different GPU architectures or library builds can still produce small floating-point differences. Evaluate the saved `best_model.pth` files when the reported predictions must remain identical.